# Fine-tuning LLM untuk Asisten Legal Tim
## Submission Proyek Akhir - PGABL

| **Informasi** | **Detail** |
|---|---|
| **Nama** | Faishal Anwar Hasyim |
| **Email** | anwarfaishal86@gmail.com |

**Model**: Qwen2.5-1.5B (Text Generation)
**Dataset**: Ichsan2895/alpaca-gpt4-indonesian
**Teknik**: QLoRA (4-bit, double quantization) + SFTTrainer

---


## 1. Instalasi Library

In [1]:
%%capture
# Install Unsloth dan dependensi untuk Google Colab
!pip install unsloth[colab-new]
!pip install --no-deps unsloth[colab-no-deps]
!pip install rouge-score

## 2. Import Library dan Setup

In [2]:
import os
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from unsloth.chat_templates import get_chat_template
import getpass

# Setup API Keys - kompatibel Colab (browser) & VS Code
def get_secret(key):
    """Ambil secret dari Colab Secrets, env var, atau input manual."""
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        val = os.environ.get(key)
        if val:
            return val
        return getpass.getpass(f"Masukkan {key}: ")

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY")

print("Libraries imported successfully!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Masukkan HF_TOKEN: ··········
Masukkan WANDB_API_KEY: ··········
Libraries imported successfully!
CUDA available: True
GPU: Tesla T4


## 3. Memuat Model Fine-tuned dari Hugging Face

In [3]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

HF_USERNAME = get_secret("HF_USERNAME")
SFT_MODEL = f"{HF_USERNAME}/qwen2.5-1.5b-pgabl-legal-sft-faishal"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
    use_rslora=False, loftq_config=None,
)

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

print(f"Model loaded: {SFT_MODEL}")
model.print_trainable_parameters()

Masukkan HF_USERNAME: ··········
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_azv6va57/tokenizer_config.json.


Model loaded: Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-sft-faishal
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4. Mempersiapkan Dataset untuk GRPO

In [4]:
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

# Format dataset untuk GRPO - membutuhkan kolom 'prompt' dan 'completion'
def format_grpo_dataset(examples):
    prompts = []
    completions = []
    for inp, out in zip(examples["input"], examples["output"]):
        conversation = [
            {"role": "system", "content": "Anda adalah asisten AI yang membantu. Berpikirlah langkah demi langkah di dalam tag <think></think>, lalu berikan jawaban akhir dalam Bahasa Indonesia."},
            {"role": "user", "content": inp.strip()},
        ]
        prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)
        completions.append(out.strip())
    return {"prompt": prompts, "completion": completions}

grpo_dataset = dataset.map(format_grpo_dataset, batched=True, remove_columns=dataset.column_names)

# Ambil subset untuk efisiensi komputasi
grpo_dataset = grpo_dataset.shuffle(seed=42).select(range(min(5000, len(grpo_dataset))))

print(f"GRPO Dataset size: {len(grpo_dataset)}")
print(f"\nContoh prompt:")
print(grpo_dataset[0]["prompt"][:300])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

GRPO Dataset size: 5000

Contoh prompt:
<|im_start|>system
Anda adalah asisten AI yang membantu. Berpikirlah langkah demi langkah di dalam tag <think></think>, lalu berikan jawaban akhir dalam Bahasa Indonesia.<|im_end|>
<|im_start|>user
Tentukan hubungan antara variabel-variabel berikut.
Umur dan kecerdasan.<|im_end|>
<|im_start|>assista


## 5. Definisi Reward Functions### 5.1 format_reward_funcReward Shaping bertahap (max +1.0):

In [5]:
def format_reward_func(completions, **kwargs):
    """
    Reward Shaping untuk format <think>...</think>:
    - Tag <think> terbuka: +0.2
    - Tag </think> tertutup: +0.3
    - Format sempurna (awal, tertutup, diikuti jawaban): +1.0
    - Penalti: -0.5 jika tag muncul lebih dari satu kali
    """
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0].get("content", "") if isinstance(completion, list) else str(completion)
        reward = 0.0

        # Hitung jumlah kemunculan tag
        think_open_count = text.count("<think>")
        think_close_count = text.count("</think>")

        # Penalti jika tag muncul lebih dari sekali
        if think_open_count > 1 or think_close_count > 1:
            reward = -0.5
        else:
            # +0.2 jika ada tag <think>
            if think_open_count == 1:
                reward += 0.2
            # +0.3 jika ada tag </think>
            if think_close_count == 1:
                reward += 0.3
            # +0.5 bonus jika format sempurna
            if text.strip().startswith("<think>") and "</think>" in text:
                after_think = text.split("</think>", 1)[-1].strip()
                if len(after_think) > 10:
                    reward = 1.0

        rewards.append(reward)
    return rewards

# Test
test_good = "<think>Mari kita analisis...</think>Jawabannya adalah bahwa hak cipta dilindungi oleh UU."
test_bad = "Jawabannya adalah hak cipta."
test_double = "<think>satu</think><think>dua</think>Jawaban"
print(f"Good format -> Reward: {format_reward_func([test_good])[0]}")
print(f"No tags    -> Reward: {format_reward_func([test_bad])[0]}")
print(f"Double tags-> Reward: {format_reward_func([test_double])[0]}")

Good format -> Reward: 1.0
No tags    -> Reward: 0.0
Double tags-> Reward: -0.5


### 5.2 reasoning_length_reward

Poin berdasarkan **panjang karakter** reasoning di dalam `<think>...</think>`:
- `< 50` karakter: `+0.2`
- `50 - 199` karakter: `+0.5`
- `>= 200` karakter: `+1.0`
- Tidak ada tag `<think>`: `0.0`


In [6]:
def reasoning_length_reward(completions, **kwargs):
    """
    Poin berdasarkan panjang karakter reasoning di dalam <think>...</think>:
    - Kurang dari 50 karakter: +0.2
    - 50 hingga 199 karakter: +0.5
    - 200 karakter atau lebih: +1.0
    - Tidak ada tag <think>: 0.0
    """
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0].get("content", "") if isinstance(completion, list) else str(completion)

        think_match = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
        if not think_match:
            rewards.append(0.0)
            continue

        thinking_text = think_match.group(1).strip()
        char_count = len(thinking_text)  # Panjang karakter, bukan kata

        if char_count < 50:
            reward = 0.2
        elif char_count < 200:
            reward = 0.5
        else:
            reward = 1.0

        rewards.append(reward)
    return rewards

# Test
test_short = "<think>Ya.</think>Jawaban singkat."  # 2 chars -> +0.2
test_medium = "<think>" + "a" * 100 + "</think>Jawaban detail."  # 100 chars -> +0.5
test_long = "<think>" + "a" * 250 + "</think>Jawaban panjang."  # 250 chars -> +1.0
test_none = "Jawaban tanpa think tag."  # no tag -> 0.0
print(f"Short reasoning (2 chars)    -> Reward: {reasoning_length_reward([test_short])[0]}")
print(f"Medium reasoning (100 chars) -> Reward: {reasoning_length_reward([test_medium])[0]}")
print(f"Long reasoning (250 chars)   -> Reward: {reasoning_length_reward([test_long])[0]}")
print(f"No think tag                 -> Reward: {reasoning_length_reward([test_none])[0]}")


Short reasoning (2 chars)    -> Reward: 0.2
Medium reasoning (100 chars) -> Reward: 0.5
Long reasoning (250 chars)   -> Reward: 1.0
No think tag                 -> Reward: 0.0


### 5.3 correctness_reward

Mengukur kesamaan jawaban model dengan reference answer menggunakan **ROUGE-L**.
Menerapkan **threshold = 0.5**: jika skor ROUGE-L >= 0.5, reward flat **+1.0**, jika di bawah = **0.0**.


In [7]:
from rouge_score import rouge_scorer

def correctness_reward(completions, **kwargs):
    """
    ROUGE-L reward dengan threshold: mengukur kesamaan antara jawaban model dan reference answer.
    Menggunakan bagian setelah </think> sebagai jawaban final.
    Jika skor ROUGE-L >= threshold (0.5), reward = +1.0, selainnya = 0.0.
    """
    references = kwargs.get("completion", [])
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    threshold = 0.5  # Threshold wajar untuk ROUGE-L
    rewards = []

    for i, completion in enumerate(completions):
        text = completion if isinstance(completion, str) else completion[0].get("content", "") if isinstance(completion, list) else str(completion)

        # Ambil bagian setelah </think> sebagai jawaban final
        if "</think>" in text:
            answer = text.split("</think>", 1)[-1].strip()
        else:
            answer = text.strip()

        if i < len(references) and references[i]:
            score = scorer.score(references[i], answer)["rougeL"].fmeasure
        else:
            score = 0.0

        # Terapkan threshold: reward +1.0 jika skor >= threshold, 0.0 jika di bawah
        reward = 1.0 if score >= threshold else 0.0
        rewards.append(reward)
    return rewards

# Test
test_completion = "<think>Analisis hukum...</think>Hak cipta adalah hak eksklusif pencipta."
test_reference = "Hak cipta merupakan hak eksklusif bagi pencipta atas karya ciptaannya."
result = correctness_reward([test_completion], completion=[test_reference])
print(f"ROUGE-L score -> Reward (threshold=0.5): {result[0]}")

test_bad = "<think>Pikir...</think>Jawabannya tidak relevan sama sekali."
result_bad = correctness_reward([test_bad], completion=[test_reference])
print(f"Bad answer -> Reward (threshold=0.5): {result_bad[0]}")


ROUGE-L score -> Reward (threshold=0.5): 1.0
Bad answer -> Reward (threshold=0.5): 0.0


### 5.4 language_reward_func

In [8]:
def language_reward_func(completions, **kwargs):
    """
    Penalti -0.5 jika model menjawab dalam bahasa Inggris.
    Reward +1.0 jika menjawab dalam bahasa Indonesia.
    """
    indonesian_indicators = [
        "adalah", "yang", "untuk", "dalam", "dengan", "tidak", "akan",
        "dari", "ini", "itu", "dapat", "pada", "oleh", "karena",
        "hukum", "undang", "pasal", "hak", "pekerja", "bahwa",
        "tersebut", "maka", "jika", "atau", "serta", "juga",
        "berdasarkan", "menurut", "sebagai", "telah", "sudah",
    ]
    english_indicators = [
        "the", "is", "are", "was", "were", "have", "has", "been",
        "will", "would", "could", "should", "this", "that", "which",
        "with", "from", "they", "their", "there", "about", "because",
        "however", "therefore", "furthermore", "according",
    ]

    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0].get("content", "") if isinstance(completion, list) else str(completion)
        if "</think>" in text:
            final_text = text.split("</think>", 1)[-1].strip().lower()
        else:
            final_text = text.strip().lower()

        if len(final_text) < 5:
            rewards.append(0.0)
            continue

        en_count = sum(1 for w in english_indicators if w in final_text)
        id_count = sum(1 for w in indonesian_indicators if w in final_text)

        if en_count > id_count and en_count >= 3:
            rewards.append(-0.5)
        elif id_count > en_count:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

# Test
test_en = "The answer is based on the regulation which states that..."
test_id = "Jawaban berdasarkan peraturan yang menyatakan bahwa dalam hukum Indonesia..."
print(f"English text -> Reward: {language_reward_func([test_en])[0]}")
print(f"Indonesian text -> Reward: {language_reward_func([test_id])[0]}")

English text -> Reward: -0.5
Indonesian text -> Reward: 1.0


## 6. Menjalankan GRPO Training

In [9]:
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

# Konfigurasi GRPO
grpo_config = GRPOConfig(
    output_dir="outputs_grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    num_generations=2,  # Dikurangi untuk mitigasi OOM
    max_completion_length=256,  # Dikurangi untuk mitigasi OOM
    max_prompt_length=512,
    max_steps=200,
    logging_steps=5,
    warmup_steps=10,
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="wandb",
    run_name="grpo-training",
    gradient_accumulation_steps=4,
    seed=3407,
)

# Setup GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args=grpo_config,
    train_dataset=grpo_dataset,
)

print("GRPOTrainer siap!")
print(f"num_generations: {grpo_config.num_generations}")
print(f"max_completion_length: {grpo_config.max_completion_length}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


GRPOTrainer siap!
num_generations: 2
max_completion_length: 256


In [10]:
print("=" * 60)
print("MEMULAI GRPO TRAINING")
print("=" * 60)

trainer.train()

print("\n" + "=" * 60)
print("GRPO TRAINING SELESAI!")
print("=" * 60)

wandb.finish()

MEMULAI GRPO TRAINING


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
wandb: Currently logged in as: catatanfaishal (catatanfaishal-universitas-islam-sultan-agung) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
5,0.000000,0.900000,0.141421,113.400000,21.000000,234.600000,0.200000,77.433334,21.000000,174.400000,0.000019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.900000,0.200000
10,0.000000,0.950000,0.070711,207.250000,148.400000,256.000000,0.700000,59.400000,46.000000,71.000000,0.000017,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.950000,0.100000
15,0.000000,1.000000,0.000000,133.100000,62.600000,193.800000,0.300000,87.900000,62.600000,115.200000,0.000015,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
20,0.000000,1.000000,0.000000,197.250000,110.800000,256.000000,0.600000,92.733334,59.600000,123.600000,0.000019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.850000,0.070711,161.200000,104.400000,217.800000,0.350000,136.483334,104.400000,177.000000,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.850000,0.215470
30,0.000000,0.900000,0.141421,124.500000,22.400000,233.200000,0.300000,59.833334,22.400000,102.800000,0.000058,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.900000,0.200000
35,0.000000,1.000000,0.000000,183.400000,97.400000,219.000000,0.500000,117.700000,97.400000,136.400000,0.000025,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
40,0.000000,0.750000,0.353553,129.250000,42.800000,191.800000,0.300000,83.250000,42.800000,128.000000,0.000044,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.750000,0.415470
45,0.000000,0.850000,0.070711,123.900000,33.000000,208.200000,0.200000,99.366669,33.000000,181.200000,0.000094,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.850000,0.215470
50,0.000000,0.950000,0.070711,161.600000,119.800000,202.200000,0.450000,89.300000,68.600000,112.000000,0.000077,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.950000,0.100000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


GRPO TRAINING SELESAI!


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▄▆▅▂▂▃▃▅▄▃▃▂▂▃▂▂▂▂▃▂▂▂█▁▂▂█▂▁▁▃▄▁▃▂▄▁▁▃▁
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▇▁▁██▁█▁▁▁▁▁▁█▁▁▁▁▁▇▁▁█▁▇▁▁█▁▇▁▄██▁▁▁▇▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,▃▄▂▃▃▃▃▃▅▃▂▃▃▁▅█▂▁▃▃▄▂▂▁▂▂▁▄▂▃▂▁▇▃▄▃▂▃▃▂
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,█▆▂▃▆▂▅▃▃▆▅▅█▅▃▅▆▁▂▁▁▁▁▁▃▂▁▅▁▄▆▂▁▂▁▂▂▃▂▂
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▃██▄▃▄▄▄▁▁▆▄█▃▅▅▅▄▂▃▆▄▅▆▃▇▇▁▄▄▄▅▃▃▂▃▃▅▃▇
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▂▂▂▂▂▂▂▃▂▂▂▂▄▃▂▄▂▁▃▂▂▂▂▁▁█▂▁▂▂▂▁▂▂▂▁▂▂▁▁
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▅▆▆▆▆▃▆▂▇▆▄▆▆▆▆▆▃▆▆▆▆▆▆▁▆▄▃▆▆█▆▃▁▂▆▂▆▆▆▆
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+28,...


## 7. Upload Model GRPO ke Hugging Face

In [11]:
GRPO_MODEL_NAME = f"{HF_USERNAME}/qwen2.5-1.5b-pgabl-legal-grpo-faishal"

print(f"Mengunggah model ke: {GRPO_MODEL_NAME}")
model.push_to_hub_merged(
    GRPO_MODEL_NAME, tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
)
print(f"\n✅ Model GRPO berhasil diunggah ke: https://huggingface.co/{GRPO_MODEL_NAME}")

Mengunggah model ke: Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal


No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal`: 100%|██████████| 1/1 [00:35<00:00, 35.66s/it]


Successfully copied all 1 files from cache to `Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...faishal/model.safetensors:   1%|          | 15.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:57<00:00, 117.77s/it]


Unsloth: Merge process complete. Saved to `/content/Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal`

✅ Model GRPO berhasil diunggah ke: https://huggingface.co/Faishal-Anwar/qwen2.5-1.5b-pgabl-legal-grpo-faishal


## 8. Test Inferensi Model GRPO (Test Case Wajib)

In [12]:
FastLanguageModel.for_inference(model)

# Test Case Wajib
test_prompt = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

messages = [
    {"role": "system", "content": "Anda adalah asisten AI legal yang membantu. Berpikirlah langkah demi langkah di dalam tag <think></think>, lalu berikan jawaban akhir dalam Bahasa Indonesia."},
    {"role": "user", "content": test_prompt},
]

# Buat input dan tambahkan token <think> sebagai seed agar model memulai reasoning
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_text += "<think>\n"  # Seed model agar memulai dengan reasoning

inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Decode hanya bagian yang di-generate (tanpa prompt)
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

# Tambahkan kembali tag <think> yang kita seed
full_response = "<think>\n" + response

print("=== Test Case Wajib: Model GRPO ===")
print(f"\nPrompt: {test_prompt}")
print(f"\nResponse:\n{full_response}")

# Verifikasi keberadaan tag <think>
if "<think>" in full_response and "</think>" in full_response:
    print("\n✅ Model berhasil menghasilkan output dengan tag <think>...</think>")
else:
    print("\n⚠️ Tag <think> tidak lengkap dalam output")


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test Case Wajib: Model GRPO ===

Prompt: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Response:
<think>
Kemungkinan Anda berhak mendapatkan uang lembur tergantung pada peraturan perusahaan Anda. Beberapa perusahaan memberikan uang lembur kepada karyawan yang tidak mungkin berkontribusi dalam pekerjaan mereka. Beberapa lainnya mungkin memberikan uang lembur untuk lembur yang diizinkan atau untuk lembur yang berkontribusi. Untuk mengetahui apakah Anda berhak mendapatkan uang lembur, Anda harus mengecek peraturan perusahaan Anda. Bisa jadi ada syarat-syarat tertentu yang harus Anda penuhi untuk mendapatkan uang lembur.
</think>

✅ Model berhasil menghasilkan output dengan tag <think>...</think>
